# ChakraTransformer — REAL VIDEO Evaluation (Kaggle T4x2)

This extends the previous static-image eval (Kvasir-SEG / CVC-ClinicDB) to the
**video datasets** actually uploaded to Kaggle:

1. `polypgen20021-video` (gokulraj324) — polyp video, likely has masks
2. `ldpolypvideowithoutpolyps` (gokulraj324) — polyp-free video (false-positive test)
3. `ldpolypvideopolyponly` (gokulraj324) — polyp-only video
4. `hyperkvasir-labeled-videos-part2-001` / `-002` (gokulrocky) — labeled video
5. `endoscene-cvc300-polyp-raw-dataset` (gokulrocky) — CVC-300 (closes a gap the
   static-image report flagged as missing)
6. `cvc-sample-video` (gokulrocky)

**Anti-fabrication rule for this notebook (per project ways-of-working):**
Every metric printed below is tagged `[MEASURED]`, `[CROSS-CHECKED ESTIMATE]`, or
`[UNAVAILABLE]`. If a dataset has no ground-truth masks, we do **not** invent a
Dice/IoU score — we report presence-rate / false-positive-rate instead, and say
so explicitly. Cell 4 (exploration) must be run and read *before* trusting
anything downstream, because we don't know these datasets' exact folder layout
yet (video files vs. pre-extracted frames, masks vs. no masks).

In [ ]:
# === CELL 1: Download everything ===
import kagglehub, os

datasets_to_get = {
    "weights":        "gokulrocky/finalmuruga-harae",
    "weights_alt":    "gokulrocky/chakratransformer-weights",
    "static_eval":    "gokulrocky/chakramodel-evaluation-datasets",
    "polypgen_video": "gokulraj324/polypgen20021-video",
    "ld_no_polyp":    "gokulraj324/ldpolypvideowithoutpolyps",
    "ld_polyp_only":  "gokulraj324/ldpolypvideopolyponly",
    "hkvasir_v2_001": "gokulrocky/hyperkvasir-labeled-videos-part2-001",
    "hkvasir_v2_002": "gokulrocky/hperkvasir-labeled-videos-part2-002",
    "cvc300":         "gokulrocky/endoscene-cvc300-polyp-raw-dataset",
    "cvc_sample_vid": "gokulrocky/cvc-sample-video",
}

paths = {}
for key, slug in datasets_to_get.items():
    try:
        p = kagglehub.dataset_download(slug)
        paths[key] = p
        print(f"[OK]   {key:16s} -> {p}")
    except Exception as e:
        print(f"[FAIL] {key:16s} ({slug}): {e}")

print()
print("Resolved paths dict `paths` ready for use below.")

In [ ]:
# === CELL 2: Load model (same checkpoint as the static-image eval) ===
import torch, sys

WP = paths.get("weights") or paths.get("weights_alt")
sys.path.insert(0, f"{WP}/src")
sys.path.insert(0, f"{WP}/src/chakra_transformer")

from chakra_transformer.transformer_segmenter import ChakraTransformerSegmenter

ckpt_path = f"{WP}/weights/chakra_transformer_best.pth"
model = ChakraTransformerSegmenter(backbone_name='vit_large_patch16_384', pretrained=False)
state = torch.load(ckpt_path, map_location='cpu', weights_only=False)
sd = state['state_dict'] if isinstance(state, dict) and 'state_dict' in state else state
missing, unexpected = model.load_state_dict(sd, strict=False)
print(f"Missing keys: {len(missing)}, Unexpected keys: {len(unexpected)}")
assert len(missing) == 0 and len(unexpected) == 0, "STOP: checkpoint does not match architecture — do not trust downstream numbers if this fails."

model = model.cuda().eval()
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model loaded. Params: {n_params:.2f}M")

In [ ]:
# === CELL 3: Preprocessing + metrics (same definitions as static-image eval, reused for frames) ===
import numpy as np
import cv2

IMG_SIZE = 384
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def preprocess(img_bgr):
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    img = img.astype(np.float32) / 255.0
    img = (img - IMAGENET_MEAN) / IMAGENET_STD
    tensor = torch.from_numpy(img.transpose(2, 0, 1)).unsqueeze(0).float()
    return tensor

@torch.no_grad()
def predict_mask(img_bgr, thresh=0.5, use_amp=True):
    # Returns binary mask (H=IMG_SIZE, W=IMG_SIZE) uint8 {0,1} and raw prob map.
    x = preprocess(img_bgr).cuda()
    if use_amp:
        with torch.amp.autocast('cuda', dtype=torch.float16):
            out = model(x)
    else:
        out = model(x)
    prob = torch.sigmoid(out.float()).squeeze().cpu().numpy()
    mask = (prob > thresh).astype(np.uint8)
    return mask, prob

def dice_iou_fbeta(pred, gt, beta=1.0, eps=1e-7):
    pred = pred.astype(bool); gt = gt.astype(bool)
    tp = np.logical_and(pred, gt).sum()
    fp = np.logical_and(pred, np.logical_not(gt)).sum()
    fn = np.logical_and(np.logical_not(pred), gt).sum()
    dice = (2*tp + eps) / (2*tp + fp + fn + eps)
    iou  = (tp + eps) / (tp + fp + fn + eps)
    precision = (tp + eps) / (tp + fp + eps)
    recall    = (tp + eps) / (tp + fn + eps)
    fbeta = ((1+beta**2) * precision * recall + eps) / (beta**2 * precision + recall + eps)
    return dice, iou, fbeta

def mask_area_frac(mask):
    return mask.sum() / mask.size

@torch.no_grad()
def predict_mask_tta(img_bgr, thresh=0.5):
    # 4-way TTA: identity, hflip, vflip, both -- same recipe as the static-image
    # "max accuracy" eval. ~4x slower than predict_mask(); use only where you
    # actually have GT masks to measure a Dice gain against.
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    img = img.astype(np.float32) / 255.0
    img = (img - IMAGENET_MEAN) / IMAGENET_STD
    base = torch.from_numpy(img.transpose(2, 0, 1)).float()

    variants = {
        "identity": base,
        "hflip": torch.flip(base, dims=[2]),
        "vflip": torch.flip(base, dims=[1]),
        "both":  torch.flip(base, dims=[1, 2]),
    }
    probs = []
    for name, v in variants.items():
        x = v.unsqueeze(0).cuda()
        with torch.amp.autocast('cuda', dtype=torch.float16):
            out = model(x)
        p = torch.sigmoid(out.float()).squeeze().cpu().numpy()
        if name == "hflip":
            p = np.flip(p, axis=1)
        elif name == "vflip":
            p = np.flip(p, axis=0)
        elif name == "both":
            p = np.flip(np.flip(p, axis=0), axis=1)
        probs.append(p)
    prob = np.mean(probs, axis=0)
    mask = (prob > thresh).astype(np.uint8)
    return mask, prob

def temporal_iou(mask_t, mask_t1):
    # IoU between consecutive-frame predictions - a no-ground-truth proxy for flicker/stability.
    inter = np.logical_and(mask_t.astype(bool), mask_t1.astype(bool)).sum()
    union = np.logical_or(mask_t.astype(bool), mask_t1.astype(bool)).sum()
    if union == 0:
        return 1.0  # both empty -> perfectly stable
    return inter / union

print("Preprocessing + metric functions defined.")

In [ ]:
# === CELL 4: EXPLORE — run this BEFORE trusting any eval below ===
# We do not yet know whether each dataset ships pre-extracted frames or raw
# video files, or whether masks exist. This cell inspects each path and
# prints what it finds so the loader in Cell 5 can be pointed correctly.

VIDEO_EXTS = ('.mp4', '.avi', '.mov', '.mkv', '.mpg', '.mpeg', '.wmv')
IMG_EXTS   = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')

def explore(root, max_depth=3, max_list=6):
    print(f"\n{'='*80}\nEXPLORING: {root}\n{'='*80}")
    if not root or not os.path.exists(root):
        print("  [MISSING] path does not exist -- download may have failed above.")
        return
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath[len(root):].count(os.sep)
        if depth > max_depth:
            dirnames[:] = []
            continue
        vids = [f for f in filenames if f.lower().endswith(VIDEO_EXTS)]
        imgs = [f for f in filenames if f.lower().endswith(IMG_EXTS)]
        others = [f for f in filenames if f not in vids and f not in imgs]
        rel = dirpath[len(root):] or "/"
        if vids or imgs or others:
            print(f"  {rel}  [{len(vids)} video, {len(imgs)} image, {len(others)} other]")
            if vids:
                print(f"      video sample: {sorted(vids)[:max_list]}")
            if imgs:
                print(f"      image sample: {sorted(imgs)[:max_list]}")
            if others and len(others) <= 10:
                print(f"      other files : {sorted(others)[:max_list]}")

for key in ["polypgen_video", "ld_no_polyp", "ld_polyp_only",
            "hkvasir_v2_001", "hkvasir_v2_002", "cvc300", "cvc_sample_vid"]:
    explore(paths.get(key))

print("\n\n>>> READ THE ABOVE before running Cell 5+.")
print(">>> Look specifically for: (a) .mp4/.avi files vs loose frame images,")
print(">>>   (b) a 'masks'/'gt'/'annotation' folder alongside images/video,")
print(">>>   (c) any labels.csv / .json / .txt annotation file.")

In [ ]:
# === CELL 5: Generic frame source ===
# Auto-detects: (a) video file -> decode with cv2.VideoCapture,
#               (b) folder of frame images -> sorted image list.
# Also tries to auto-pair a mask if a sibling 'mask'/'gt'/'label' dir or
# same-stem file exists. If none found, mask_path is None for every frame
# (segmentation metrics are then reported as [UNAVAILABLE], never guessed).

from pathlib import Path

def find_mask_dir(images_dir):
    p = Path(images_dir)
    for cand in [p.parent / "masks", p.parent / "mask", p.parent / "gt",
                 p.parent / "annotations", p.parent / "labels"]:
        if cand.exists():
            return cand
    return None

def iter_video_frames(video_path, stride=1):
    cap = cv2.VideoCapture(str(video_path))
    idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if idx % stride == 0:
            yield idx, frame
        idx += 1
    cap.release()

def iter_image_folder_frames(images_dir, mask_dir=None, stride=1):
    imgs = sorted([p for p in Path(images_dir).iterdir() if p.suffix.lower() in IMG_EXTS])
    for i, img_path in enumerate(imgs):
        if i % stride != 0:
            continue
        frame = cv2.imread(str(img_path))
        if frame is None:
            continue
        mask = None
        if mask_dir is not None:
            for ext in IMG_EXTS:
                cand = Path(mask_dir) / (img_path.stem + ext)
                if cand.exists():
                    m = cv2.imread(str(cand), cv2.IMREAD_GRAYSCALE)
                    if m is not None:
                        m = cv2.resize(m, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
                        mask = (m > 127).astype(np.uint8)
                    break
        yield i, frame, mask

def discover_sources(root, max_depth=4):
    # Walk root, return list of dicts: {type: 'video'|'frames', path, mask_dir}
    sources = []
    if not root or not os.path.exists(root):
        return sources
    seen_frame_dirs = set()
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath[len(root):].count(os.sep)
        if depth > max_depth:
            dirnames[:] = []
            continue
        vids = [f for f in filenames if f.lower().endswith(VIDEO_EXTS)]
        for v in vids:
            sources.append({"type": "video", "path": os.path.join(dirpath, v), "mask_dir": None})
        imgs = [f for f in filenames if f.lower().endswith(IMG_EXTS)]
        low = os.path.basename(dirpath).lower()
        if imgs and low not in ("masks", "mask", "gt", "annotations", "labels") and dirpath not in seen_frame_dirs:
            mdir = find_mask_dir(dirpath)
            sources.append({"type": "frames", "path": dirpath, "mask_dir": mdir, "n_imgs": len(imgs)})
            seen_frame_dirs.add(dirpath)
    return sources

print("Frame-source discovery functions ready. Run Cell 6+ to use them.")

In [ ]:
# === CELL 6: Evaluate a video/frame source set. Reused for every dataset below. ===
# thresh follows the same 0.5 default used in the static-image eval for comparability.

from tqdm.auto import tqdm

def eval_source_set(name, sources, stride=5, max_frames_per_source=300, thresh=0.5, use_tta=False):
    # stride: only run every Nth frame (video is redundant frame-to-frame; this
    #         keeps runtime sane on Kaggle's session limit -- report this stride
    #         explicitly, do not silently subsample and imply full coverage).
    # use_tta: True -> 4-way TTA (max-accuracy mode, ~4x slower). False -> single
    #          pass FP16 (max-fps-friendly mode). Run both on GT-bearing sources
    #          if you want an honest accuracy/speed tradeoff comparison on video,
    #          the same way the static-image report compared baseline vs +TTA.
    # Returns a result dict with tier-labeled fields.
    dices, ious, fbs = [], [], []
    positive_frame_flags = []   # 1 if predicted mask has any polyp pixel, else 0
    temporal_ious = []
    n_frames_seen = 0
    n_frames_with_gt = 0
    prev_mask = None
    predict_fn = predict_mask_tta if use_tta else predict_mask

    for src in sources:
        if src["type"] == "video":
            gen = iter_video_frames(src["path"], stride=stride)
            gen = ((i, f, None) for i, f in gen)
        else:
            gen = iter_image_folder_frames(src["path"], mask_dir=src["mask_dir"], stride=stride)

        count_this_source = 0
        for idx, frame, gt_mask in tqdm(gen, desc=f"{name}:{os.path.basename(src['path'])}", leave=False):
            if count_this_source >= max_frames_per_source:
                break
            pred_mask, prob = predict_fn(frame, thresh=thresh)
            n_frames_seen += 1
            count_this_source += 1

            positive_frame_flags.append(1 if pred_mask.sum() > 0 else 0)

            if prev_mask is not None:
                temporal_ious.append(temporal_iou(prev_mask, pred_mask))
            prev_mask = pred_mask

            if gt_mask is not None:
                n_frames_with_gt += 1
                d, iou, fb = dice_iou_fbeta(pred_mask, gt_mask)
                dices.append(d); ious.append(iou); fbs.append(fb)

    result = {
        "name": name,
        "n_frames_seen": n_frames_seen,
        "n_frames_with_gt": n_frames_with_gt,
        "stride": stride,
        "mode": "TTA (max-accuracy)" if use_tta else "single-pass FP16 (max-fps-friendly)",
        "thresh": thresh,
    }
    if n_frames_with_gt > 0:
        result["dice_MEASURED"] = float(np.mean(dices))
        result["iou_MEASURED"] = float(np.mean(ious))
        result["fbeta_MEASURED"] = float(np.mean(fbs))
    else:
        result["dice_MEASURED"] = None
        result["note"] = "[UNAVAILABLE] no ground-truth masks found for this source -- Dice/IoU NOT computed, not estimated."

    result["positive_rate"] = float(np.mean(positive_frame_flags)) if positive_frame_flags else None
    result["mean_temporal_iou"] = float(np.mean(temporal_ious)) if temporal_ious else None
    return result

print("eval_source_set() ready.")

In [ ]:
# === CELL 6b: THRESHOLD SWEEP (max-accuracy tuning step) ===
# The static-image report used a fixed thresh=0.5 without ever checking whether
# that's actually optimal for this checkpoint. This sweeps thresh on whichever
# GT-bearing dataset is available (checked in this priority order) using a
# small, fast subset, then reports the best Dice threshold to feed into Cell 7.
# If NO dataset here turns out to have masks, this cell prints [UNAVAILABLE]
# and Cell 7 falls back to thresh=0.5 -- it does not invent a "tuned" value.

SWEEP_CANDIDATES = ["cvc300", "polypgen_video", "ld_polyp_only", "hkvasir_v2_001", "hkvasir_v2_002"]
SWEEP_THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7]

sweep_source_key = None
sweep_sources = None
for key in SWEEP_CANDIDATES:
    s = discover_sources(paths.get(key))
    probe = eval_source_set(f"{key}_probe", s, stride=10, max_frames_per_source=40)
    if probe["n_frames_with_gt"] > 0:
        sweep_source_key = key
        sweep_sources = s
        print(f"Using '{key}' for threshold sweep ({probe['n_frames_with_gt']} GT frames found in probe).")
        break

if sweep_source_key is None:
    print("[UNAVAILABLE] No dataset with ground-truth masks found for threshold sweep.")
    print("Falling back to thresh=0.5 in Cell 7 -- not tuned, just the untested default.")
    best_thresh = 0.5
else:
    sweep_results = {}
    for th in SWEEP_THRESHOLDS:
        r = eval_source_set(f"sweep_{th}", sweep_sources, stride=8, max_frames_per_source=80, thresh=th)
        sweep_results[th] = r["dice_MEASURED"]
        print(f"  thresh={th}: [MEASURED] Dice={r['dice_MEASURED']:.4f}  (n_gt={r['n_frames_with_gt']})")
    best_thresh = max(sweep_results, key=sweep_results.get)
    print(f"\n[MEASURED] Best threshold on '{sweep_source_key}' subset: {best_thresh} (Dice={sweep_results[best_thresh]:.4f})")
    print("NOTE: this was tuned on a small subset (fast sweep) -- re-verify on the")
    print("      full run in Cell 7 rather than trusting this number as final.")

In [ ]:
# === CELL 7: Run on each dataset -- BOTH modes where GT exists. EDIT the
# paths.get(...) keys if Cell 4's exploration output showed a different
# top-level folder name than expected. ===
#
#   "single-pass FP16"  = max-fps-friendly mode (thresh tuned in Cell 6b)
#   "TTA"                = max-accuracy mode (4-way flip TTA, ~4x slower)
#
# TTA is only run on sources where Cell 4/6b confirmed GT masks exist --
# there is no point spending 4x compute on a source where we can't measure
# whether it actually helped.

results = {}

def run_both_modes(key, stride, max_frames=300):
    src = discover_sources(paths.get(key))
    print(f"{key}: found {len(src)} source(s) -> {[s['type'] for s in src][:10]}")
    baseline = eval_source_set(key, src, stride=stride, max_frames_per_source=max_frames, thresh=best_thresh, use_tta=False)
    results[key] = baseline
    if baseline["n_frames_with_gt"] > 0:
        tta = eval_source_set(key + "_TTA", src, stride=stride, max_frames_per_source=max_frames, thresh=best_thresh, use_tta=True)
        results[key + "_TTA"] = tta
        gain = (tta["dice_MEASURED"] - baseline["dice_MEASURED"]) if (tta["dice_MEASURED"] and baseline["dice_MEASURED"]) else None
        if gain is not None:
            print(f"  [MEASURED] TTA Dice gain on {key}: {gain:+.4f} (baseline {baseline['dice_MEASURED']:.4f} -> TTA {tta['dice_MEASURED']:.4f})")

# 1) PolypGen video -- most likely to have masks (it's a segmentation benchmark)
run_both_modes("polypgen_video", stride=5)

# 2) LD-PolypVideo WITHOUT polyps -- false-positive test, single-pass only
#    (no GT masks by definition -- TTA has nothing to measure a Dice gain against).
src = discover_sources(paths.get("ld_no_polyp"))
print(f"ld_no_polyp: found {len(src)} source(s) -> {[s['type'] for s in src][:10]}")
results["ld_no_polyp"] = eval_source_set("ld_no_polyp", src, stride=5, thresh=best_thresh)

# 3) LD-PolypVideo POLYP-ONLY -- LDPolypVideo's public release ships bounding-box
#    labels, not pixel masks, in most versions. run_both_modes() only fires TTA
#    if Cell 4/6b confirmed a masks/ dir actually exists here.
run_both_modes("ld_polyp_only", stride=5)

# 4/5) HyperKvasir labeled videos
for key in ["hkvasir_v2_001", "hkvasir_v2_002"]:
    run_both_modes(key, stride=5)

# 6) CVC-300 -- closes the gap the static-image report flagged as missing.
#    Static-frame benchmark dataset, so stride=1 (no redundant frames to skip)
#    for a full, standard-comparable score.
run_both_modes("cvc300", stride=1, max_frames=1000)

# 7) CVC sample video
run_both_modes("cvc_sample_vid", stride=5)

print(f"\n\nTuned threshold used throughout: {best_thresh}")
print("\n=== RAW RESULTS DICT ===")
for k, v in results.items():
    print(k, "->", v)

In [ ]:
# === CELL 8: REAL video FPS benchmark -- three configs, decode+preprocess+
# inference on actual video, not synthetic random tensors like the
# static-image benchmark used:
#   1. Single-frame FP32   (worst case, sanity floor)
#   2. Single-frame FP16   (what Cell 7's baseline mode effectively runs at)
#   3. Batched FP16, dual-GPU DataParallel (max-FPS mode, mirrors the
#      "Dual T4 FP16 bs=16 -> 46.61 FPS" claim from the static-image report,
#      but measured on real decoded video frames instead of torch.randn())
import time

def bench_real_video_fps(video_path, n_frames=150, warmup=15, use_amp=True):
    cap = cv2.VideoCapture(str(video_path))
    times = []
    i = 0
    while i < n_frames + warmup:
        ok, frame = cap.read()
        if not ok:
            cap.set(cv2.CAP_PROP_POS_FRAMES, 0)  # loop short videos
            continue
        torch.cuda.synchronize()
        t0 = time.time()
        _ = predict_mask(frame, use_amp=use_amp)
        torch.cuda.synchronize()
        t1 = time.time()
        if i >= warmup:
            times.append(t1 - t0)
        i += 1
    cap.release()
    times = np.array(times)
    fps = 1.0 / times.mean()
    return fps, times.mean() * 1000

@torch.no_grad()
def bench_real_video_fps_batched(video_path, batch_size=16, n_batches=20, warmup=3):
    # Requires 2 GPUs to actually exercise DataParallel; on a single-GPU
    # session this still runs (just without the dual-GPU speedup) so the
    # cell doesn't hard-fail on a 1xT4 kernel.
    n_gpus = torch.cuda.device_count()
    model_dp = torch.nn.DataParallel(model, device_ids=list(range(n_gpus))) if n_gpus > 1 else model
    cap = cv2.VideoCapture(str(video_path))
    times = []
    for b in range(n_batches + warmup):
        frames = []
        for _ in range(batch_size):
            ok, frame = cap.read()
            if not ok:
                cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                ok, frame = cap.read()
            frames.append(preprocess(frame).squeeze(0))
        x = torch.stack(frames).cuda()
        torch.cuda.synchronize()
        t0 = time.time()
        with torch.amp.autocast('cuda', dtype=torch.float16):
            _ = model_dp(x)
        torch.cuda.synchronize()
        t1 = time.time()
        if b >= warmup:
            times.append(t1 - t0)
    cap.release()
    times = np.array(times)
    fps = batch_size / times.mean()
    return fps, times.mean() * 1000, n_gpus

# pick the first real video file found across the discovered sources
video_candidates = []
for key in ["polypgen_video", "ld_no_polyp", "ld_polyp_only", "cvc_sample_vid"]:
    for s in discover_sources(paths.get(key)):
        if s["type"] == "video":
            video_candidates.append(s["path"])

if video_candidates:
    vp = video_candidates[0]
    fps_fp16, ms_fp16 = bench_real_video_fps(vp, use_amp=True)
    fps_fp32, ms_fp32 = bench_real_video_fps(vp, use_amp=False)
    fps_batched, ms_batched, n_gpus = bench_real_video_fps_batched(vp, batch_size=16)
    print(f"[MEASURED] Real video decode+preprocess+inference FPS on {os.path.basename(vp)}:")
    print(f"  Single-frame FP32:               {fps_fp32:.2f} FPS ({ms_fp32:.2f} ms/frame)")
    print(f"  Single-frame FP16:                {fps_fp16:.2f} FPS ({ms_fp16:.2f} ms/frame)")
    print(f"  Batched FP16, bs=16, {n_gpus} GPU(s):  {fps_batched:.2f} FPS ({ms_batched:.2f} ms/batch)")
    if n_gpus < 2:
        print("  NOTE: only 1 GPU visible this session -- for the dual-T4 speedup,")
        print("        confirm the Kaggle accelerator is set to 'GPU T4 x2'.")
    print(f"  FP16: {fps_fp16:.2f} FPS ({ms_fp16:.2f} ms/frame)")
    print(f"  FP32: {fps_fp32:.2f} FPS ({ms_fp32:.2f} ms/frame)")
    print("  NOTE: single-frame, single-GPU, includes cv2 decode + preprocess overhead")
    print("        (this is why it will likely be LOWER than the synthetic-tensor")
    print("        FPS numbers from the static-image benchmark -- that's expected")
    print("        and is the more honest number for real deployment.)")
else:
    print("[UNAVAILABLE] No actual video file (.mp4/.avi/...) found in any discovered "
          "source -- if all video datasets turned out to be pre-extracted frame folders, "
          "this real-video-decode FPS benchmark cannot run. Use Cell 4's output to confirm.")

In [ ]:
# === CELL 9: Write final tier-labeled report (no fabricated numbers) ===

report_lines = []
report_lines.append("="*80)
report_lines.append("CHAKRA-TRANSFORMER -- REAL VIDEO EVALUATION REPORT")
report_lines.append("Hardware: Kaggle T4 x2  |  Model: chakra_transformer_best.pth (309.17M params)")
report_lines.append("="*80)
report_lines.append("")
report_lines.append("Tier legend: [MEASURED] = computed this session from real frames/masks.")
report_lines.append("             [UNAVAILABLE] = no ground truth existed to compute this -- not estimated.")
report_lines.append("")
report_lines.append(f"Threshold used: {best_thresh} (tuned in Cell 6b on '{sweep_source_key}'; falls back to 0.5 if [UNAVAILABLE])")
report_lines.append("Two modes per GT-bearing source: 'single-pass FP16' (max-fps-friendly) vs 'TTA' (max-accuracy).")
report_lines.append("")

for k, v in results.items():
    report_lines.append(f"--- {k}  [{v.get('mode', '?')}] ---")
    report_lines.append(f"  frames seen (stride={v['stride']}): {v['n_frames_seen']}")
    report_lines.append(f"  frames with GT mask: {v['n_frames_with_gt']}")
    if v["dice_MEASURED"] is not None:
        report_lines.append(f"  [MEASURED] Dice={v['dice_MEASURED']:.4f}  IoU={v['iou_MEASURED']:.4f}  Fbeta={v['fbeta_MEASURED']:.4f}")
    else:
        report_lines.append(f"  {v.get('note', '[UNAVAILABLE] no GT masks')}")
    if v["positive_rate"] is not None:
        report_lines.append(f"  [MEASURED] positive-frame rate (any predicted polyp pixel): {v['positive_rate']*100:.1f}%")
    if v["mean_temporal_iou"] is not None:
        report_lines.append(f"  [MEASURED] mean frame-to-frame temporal IoU (stability proxy): {v['mean_temporal_iou']:.4f}")
    report_lines.append("")

report_lines.append("Interpretation notes (fill in after reading numbers above):")
report_lines.append("  - ld_no_polyp positive_rate is your real-world false-positive rate on")
report_lines.append("    polyp-free footage -- this is arguably the most clinically important")
report_lines.append("    number in this whole notebook and has no static-image equivalent.")
report_lines.append("  - Low mean_temporal_iou on polyp-containing video = flickering masks")
report_lines.append("    frame-to-frame, a real deployment problem invisible in static-image Dice.")
report_lines.append("  - Any dataset reporting [UNAVAILABLE] for Dice needs a manual check of")
report_lines.append("    whether masks genuinely don't exist, or whether Cell 5's mask-pairing")
report_lines.append("    logic just didn't find them (check Cell 4 output for folder naming).")
report_lines.append("  - Compare '<key>' vs '<key>_TTA' rows above for the real accuracy/speed")
report_lines.append("    tradeoff on video -- if the TTA gain is small, it's not worth the ~4x")
report_lines.append("    slowdown for a real-time deployment; if it's large, it may be worth it")
report_lines.append("    only for an offline/paper-mode pipeline, per Cell 8's FPS numbers.")

full_report = "\n".join(report_lines)
print(full_report)

os.makedirs("/kaggle/working/results", exist_ok=True)
with open("/kaggle/working/results/video_evaluation_report.md", "w") as f:
    f.write(full_report)
import json as _json
with open("/kaggle/working/results/video_evaluation_results.json", "w") as f:
    _json.dump(results, f, indent=2)
print("\nSaved to /kaggle/working/results/video_evaluation_report.md and .json")